# User and Case Spec Stats

Statistics for the current single/composite benchmark schema. Run `data/benchmark_builder/build_user_specs.py` and `data/benchmark_builder/build_user_cards.py` first if the JSON files are missing or stale.

In [1]:
from collections import Counter
from pathlib import Path
import json
import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'data' / 'benchmark' / 'case_specs.json').exists():
            return path
    raise FileNotFoundError('Could not find data/benchmark/case_specs.json from current working directory or parents.')

repo_root = find_repo_root()
case_specs_path = repo_root / 'data' / 'benchmark' / 'case_specs.json'
user_cards_path = repo_root / 'data' / 'benchmark' / 'user_cards.json'
specs = json.loads(case_specs_path.read_text())
cards = json.loads(user_cards_path.read_text()) if user_cards_path.exists() else []
len(specs), len(cards)

(300, 300)

In [2]:
def location_kind(location_requirement):
    for key in ['counties', 'cities', 'zipcodes']:
        if location_requirement.get(key):
            return key
    return 'none'

def schedule_kind(schedule):
    if schedule.get('requires_24_hours'):
        return '24hr'
    return schedule.get('time', 'any')

def schedule_day(schedule):
    return '24hr' if schedule.get('requires_24_hours') else schedule.get('day', 'none')

def need_rows(specs):
    rows = []
    for spec in specs:
        for need in spec['needs']:
            rows.append({
                'case_id': spec['case_id'],
                'case_type': spec['case_type'],
                'need_id': need['need_id'],
                'service_category': need['service_categories'][0],
                'location_kind': location_kind(spec['location_requirement']),
                'schedule_kind': schedule_kind(need['schedule']),
                'schedule_day': schedule_day(need['schedule']),
                'ground_truth_resource_id': need['ground_truth_resource_id'],
            })
    return rows

def gt_rows(specs):
    rows = []
    for spec in specs:
        for gt in spec['ground_truth_resources']:
            rows.append({
                'case_id': spec['case_id'],
                'case_type': spec['case_type'],
                'resource_id': gt['resource_id'],
                'intake_method_count': len(gt.get('intake_methods') or []),
                'document_requirement_count': len(gt.get('document_requirements') or []),
            })
    return rows

cases_df = pd.DataFrame([{
    'case_id': spec['case_id'],
    'case_type': spec['case_type'],
    'need_count': len(spec['needs']),
    'gt_count': len(spec['ground_truth_resources']),
    'location_kind': location_kind(spec['location_requirement']),
} for spec in specs])
needs_df = pd.DataFrame(need_rows(specs))
gt_df = pd.DataFrame(gt_rows(specs))
cases_df.head(), needs_df.head(), gt_df.head()

(    case_id  case_type  need_count  gt_count location_kind
 0  spec-001  composite           2         2      counties
 1  spec-002  composite           2         2      zipcodes
 2  spec-003     single           1         1        cities
 3  spec-004     single           1         1      zipcodes
 4  spec-005     single           1         1      zipcodes,
     case_id  case_type need_id              service_category location_kind  \
 0  spec-001  composite  need_1                          Food      counties   
 1  spec-001  composite  need_2       Pet and Animal Services      counties   
 2  spec-002  composite  need_1  Family and Caregiver Support      zipcodes   
 3  spec-002  composite  need_2                    Legal Help      zipcodes   
 4  spec-003     single  need_1      Police and Public Safety        cities   
 
   schedule_kind schedule_day                       ground_truth_resource_id  
 0       morning          sat     in211-11221-75127-food-and-clothing-pantry  
 1   

In [3]:
cases_df['case_type'].value_counts().rename_axis('case_type').reset_index(name='cases')

,case_type,cases
0,composite,150
1,single,150


In [4]:
pd.crosstab(cases_df['case_type'], cases_df['location_kind'], normalize='index').round(3)

location_kind,cities,counties,zipcodes
case_type,,,
composite,0.313,0.180,0.507
single,0.233,0.153,0.613


In [5]:
pd.crosstab(needs_df['case_type'], needs_df['schedule_kind'], normalize='index').round(3)

schedule_kind,24hr,afternoon,any,morning
case_type,,,,
composite,0.010,0.303,0.320,0.367
single,0.007,0.367,0.333,0.293


In [6]:
pd.crosstab(needs_df['case_type'], needs_df['schedule_day'], normalize='index').round(3)

schedule_day,24hr,fri,mon,sat,sun,thu,tue,wed
case_type,,,,,,,,
composite,0.010,0.20,0.150,0.123,0.077,0.160,0.133,0.147
single,0.007,0.16,0.153,0.153,0.060,0.187,0.153,0.127


In [7]:
service_dist = pd.crosstab(needs_df['service_category'], needs_df['case_type'])
service_dist['single_rate'] = service_dist.get('single', 0) / max((needs_df['case_type'] == 'single').sum(), 1)
service_dist['composite_rate'] = service_dist.get('composite', 0) / max((needs_df['case_type'] == 'composite').sum(), 1)
service_dist['pct_point_gap'] = service_dist['composite_rate'] - service_dist['single_rate']
service_dist.sort_values('pct_point_gap', key=lambda s: s.abs(), ascending=False).head(20)

case_type,composite,single,single_rate,composite_rate,pct_point_gap
service_category,,,,,
Food,32,8,0.053333,0.106667,0.053333
Emergency and Hospital Care,2,5,0.033333,0.006667,-0.026667
Information and Referral,37,15,0.100000,0.123333,0.023333
Utility Bill Help,16,5,0.033333,0.053333,0.020000
Pet and Animal Services,6,6,0.040000,0.020000,-0.020000
Substance Use Help,8,7,0.046667,0.026667,-0.020000
Government Offices and Public Services,13,4,0.026667,0.043333,0.016667
Housing and Shelter,10,7,0.046667,0.033333,-0.013333
Business and Community Development,0,2,0.013333,0.000000,-0.013333


In [8]:
gt_df.groupby('case_type')[['intake_method_count', 'document_requirement_count']].agg(['min', 'mean', 'max'])

intake_method_count               document_requirement_count  \
                          min      mean max                        min   
case_type                                                                
composite                   1  1.873333   4                          0   
single                      1  1.726667   4                          0   

                         
               mean max  
case_type                
composite  0.736667   6  
single     0.826667   6

In [9]:
if cards:
    card_df = pd.DataFrame([{
        'case_id': card['case_id'],
        'case_type': card['case_type'],
        'trait': (card.get('traits') or [''])[0],
        'opening_chars': len(card.get('opening') or ''),
        'need_count': len(card.get('needs') or []),
    } for card in cards])
    card_df.head()
else:
    card_df = pd.DataFrame()
card_df.shape

(300, 5)

In [10]:
pd.crosstab(card_df['case_type'], card_df['trait']) if not card_df.empty else pd.DataFrame()

trait,impatience,incomplete_answer,inconsistency,normal,rambling,unreasonable_demand
case_type,,,,,,
composite,25,25,25,25,25,25
single,25,25,25,25,25,25


In [11]:
card_df.groupby(['case_type', 'trait'])['opening_chars'].agg(['count', 'mean', 'min', 'max']).round(1) if not card_df.empty else pd.DataFrame()

count   mean  min  max
case_type trait                                      
composite impatience              25  182.4  119  264
          incomplete_answer       25  181.4  144  217
          inconsistency           25  167.2  105  224
          normal                  25  180.9  137  221
          rambling                25  188.0  107  299
          unreasonable_demand     25  181.6  120  237
single    impatience              25  118.4   72  180
          incomplete_answer       25  115.2   69  211
          inconsistency           25  113.6   63  226
          normal                  25  132.3   68  190
          rambling                25  112.7   73  176
          unreasonable_demand     25  151.8  103  216